# GROUPDNA

## WhatsApp Group Chat Analytics using Python and NumPy

### Minor Project

**Submitted By:** Bhawesh Pandey  
**Roll Number:** 230602  
**Course:** B.Tech CSE (AI & ML)

---

## Project Objective

The objective of this project is to analyze a WhatsApp group chat using Python fundamentals and NumPy. The program extracts useful insights such as group statistics, participant activity, response behaviour, word frequency, activity heatmap and personality archetypes from the chat data.

The project has been developed by following the given project guidelines and without using high-level data analysis libraries.

# Importing libraries

In [7]:
# ==========================================
# Import Required Libraries
# ==========================================

# NumPy is used for creating the activity heatmap
import numpy as np

# datetime is used for working with dates and time
from datetime import datetime

# Reading the Dataset

In [8]:
# ==========================================
# Reading the WhatsApp Chat File
# ==========================================

# Name of the chat file
file_name = "hostel_bois.txt"

# Open the file in read mode with UTF-8 encoding
with open(file_name, "r", encoding="utf-8") as file:
    chat_lines = file.readlines()

# Display total number of lines in the file
print("Total Lines in File :", len(chat_lines))

Total Lines in File : 3178


# Checking the file

In [6]:
# Display first five lines to verify that the file
# has been read correctly

for i in range(5):
    print(chat_lines[i].strip())

01/04/24, 01:12 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
01/04/24, 01:13 - Priya created group "Hostel Bois 4ever"
01/04/24, 01:14 - Priya added you
01/04/24, 01:17 - Rahul: scene fix
01/04/24, 01:17 - Rahul: haan


# Feature 1 : Chat Parser

## Objective

The objective of this feature is to read the WhatsApp chat file and extract useful information from every message. The parser will identify the date, time, sender name and message while handling different edge cases such as system messages, deleted messages, media messages and multiline messages.

In [9]:
# ==========================================================
# Feature 1 : Chat Parser
# Step 1 - Detect New Messages
# ==========================================================

# This function checks whether the current line
# starts with a valid WhatsApp timestamp.

def is_new_message(line):

    try:
        datetime.strptime(line[:14], "%d/%m/%y, %H:%M")
        return True

    except:
        return False

### Verification

The following code verifies whether the function correctly identifies a new WhatsApp message.

In [10]:
# Test the function

print(is_new_message(chat_lines[0]))
print(is_new_message(chat_lines[4]))
print(is_new_message("hello everyone"))

True
True
False


##  Parse the WhatsApp Chat

### Objective

In this step, we will extract useful information from every WhatsApp message. Each valid message will be divided into four parts:

- Date
- Time
- Sender Name
- Message

The parser will also handle multiline messages by joining them with the previous message.

In [13]:
# ==========================================================
# Feature 1 : Chat Parser
# ==========================================================

# This function checks whether a line starts
# with a valid WhatsApp timestamp.

def is_new_message(line):

    try:
        datetime.strptime(line[:14], "%d/%m/%y, %H:%M")
        return True

    except:
        return False


# List to store all parsed chat messages
parsed_messages = []

# Variable to temporarily store the current message
current_message = None

# Read every line from the dataset
for line in chat_lines:

    # Remove extra spaces and newline characters
    line = line.strip()

    # Skip empty lines
    if line == "":
        continue

    # Check whether the line starts with a new message
    if is_new_message(line):

        # Save the previous message before reading a new one
        if current_message is not None:
            parsed_messages.append(current_message)

        # Split date-time and remaining text
        first_split = line.split(" - ", 1)

        date_time = first_split[0]
        remaining_text = first_split[1]

        # Separate date and time
        date_part, time_part = date_time.split(", ")

        # Convert into datetime object
        date_time_object = datetime.strptime(
            date_part + " " + time_part,
            "%d/%m/%y %H:%M"
        )

        # Check whether the message contains sender name
        if ": " in remaining_text:

            sender, message = remaining_text.split(": ", 1)

        else:

            # System message
            sender = "System"
            message = remaining_text

        # Store the extracted information
        current_message = {
            "datetime": date_time_object,
            "date": date_part,
            "time": time_part,
            "sender": sender,
            "message": message
        }

    else:

        # Attach multiline messages to the previous message
        if current_message is not None:
            current_message["message"] += "\n" + line

# Store the last message
if current_message is not None:
    parsed_messages.append(current_message)

##  Verification

The following code checks whether the parser has successfully extracted the messages from the dataset.

In [12]:
# ==========================================================
# Verify Parsed Messages
# ==========================================================

print("Total Parsed Messages :", len(parsed_messages))

print("\nFirst Parsed Message\n")

print(parsed_messages[0])

print("\nSecond Parsed Message\n")

print(parsed_messages[1])

print("\nLast Parsed Message\n")

print(parsed_messages[-1])

Total Parsed Messages : 3178

First Parsed Message

{'date': '01/04/24', 'time': '01:12', 'sender': 'System', 'message': 'Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.'}

Second Parsed Message

{'date': '01/04/24', 'time': '01:13', 'sender': 'System', 'message': 'Priya created group "Hostel Bois 4ever"'}

Last Parsed Message

{'date': '30/05/24', 'time': '23:31', 'sender': 'Aman', 'message': 'anyone awake?'}


## Observation

The parser successfully reads the WhatsApp dataset and extracts each chat entry into a dictionary. Every dictionary contains the date, time, sender name, message and datetime object. System messages are also identified correctly, while multiline messages are combined with their respective previous messages.

# Feature 2 : Group Overview

## Objective

The objective of this feature is to calculate the overall statistics of the WhatsApp group. It provides a summary of the chat by calculating the total number of messages, participants, media messages, deleted messages, system messages and the duration of the conversation.

In [14]:
# ==========================================================
# Feature 2 : Group Overview
# ==========================================================

# Variables used to store different statistics
total_entries = len(parsed_messages)
system_messages = 0
deleted_messages = 0
media_messages = 0

# Set is used to store unique participant names
participants = set()

# Traverse through every parsed message
for message in parsed_messages:

    sender = message["sender"]
    text = message["message"]

    # Count system messages
    if sender == "System":
        system_messages += 1

    else:
        participants.add(sender)

    # Count deleted messages
    if text == "This message was deleted":
        deleted_messages += 1

    # Count media messages
    if "<Media omitted>" in text:
        media_messages += 1

# Total user messages
user_messages = total_entries - system_messages

# Total participants
total_participants = len(participants)

In [15]:
# ==========================================================
# Chat Duration
# ==========================================================

# First and last chat date
start_date = parsed_messages[0]["datetime"]
end_date = parsed_messages[-1]["datetime"]

# Total duration of chat
chat_duration = end_date - start_date

In [16]:
# ==========================================================
# Display Group Overview
# ==========================================================

print("=" * 45)
print("          GROUP OVERVIEW")
print("=" * 45)

print(f"Total Parsed Entries      : {total_entries}")
print(f"User Messages             : {user_messages}")
print(f"System Messages           : {system_messages}")
print(f"Media Messages            : {media_messages}")
print(f"Deleted Messages          : {deleted_messages}")
print(f"Total Participants        : {total_participants}")

print(f"\nChat Started On           : {start_date.strftime('%d-%m-%Y')}")
print(f"Chat Ended On             : {end_date.strftime('%d-%m-%Y')}")

print(f"Chat Duration             : {chat_duration.days} Days")

print("=" * 45)

          GROUP OVERVIEW
Total Parsed Entries      : 3178
User Messages             : 3174
System Messages           : 4
Media Messages            : 32
Deleted Messages          : 15
Total Participants        : 6

Chat Started On           : 01-04-2024
Chat Ended On             : 30-05-2024
Chat Duration             : 59 Days


## Observation

The Group Overview provides a summary of the complete WhatsApp dataset. It shows the total number of chat entries, participant count, deleted messages, media messages and the overall duration of the conversation. These statistics provide a basic understanding of the dataset before performing detailed analysis.

# Feature 3 : Participant Statistics

## Objective

The objective of this feature is to analyze the contribution of each participant in the WhatsApp group. It calculates the total number of messages sent by every participant, identifies the most active and least active participants, and calculates the percentage contribution of each participant to the overall conversation.

In [17]:
# ==========================================================
# Feature 3 : Participant Statistics
# ==========================================================

# Dictionary to store the number of messages sent by each participant
participant_message_count = {}

# Traverse through every parsed message
for message in parsed_messages:

    sender = message["sender"]

    # Ignore system messages
    if sender == "System":
        continue

    # Add new participant to dictionary
    if sender not in participant_message_count:
        participant_message_count[sender] = 0

    # Increase message count
    participant_message_count[sender] += 1

In [18]:
# ==========================================================
# Display Messages Sent By Each Participant
# ==========================================================

print("=" * 50)
print("        PARTICIPANT MESSAGE COUNT")
print("=" * 50)

for participant in participant_message_count:
    print(f"{participant:<15} : {participant_message_count[participant]}")

print("=" * 50)

        PARTICIPANT MESSAGE COUNT
Rahul           : 953
Priya           : 718
Karan           : 354
Neha            : 635
Aman            : 490
Vikas           : 24


In [19]:
# ==========================================================
# Find Most Active and Least Active Participant
# ==========================================================

most_active = ""
least_active = ""

highest_messages = -1
lowest_messages = float("inf")

# Find maximum and minimum message count
for participant in participant_message_count:

    count = participant_message_count[participant]

    if count > highest_messages:
        highest_messages = count
        most_active = participant

    if count < lowest_messages:
        lowest_messages = count
        least_active = participant

In [20]:
# ==========================================================
# Calculate Percentage Contribution
# ==========================================================

participant_percentage = {}

for participant in participant_message_count:

    percentage = (
        participant_message_count[participant] /
        user_messages
    ) * 100

    participant_percentage[participant] = percentage

In [22]:
# ==========================================================
# Display Participant Statistics
# ==========================================================

# Sort participants according to message count (Highest to Lowest)
sorted_participants = sorted(
    participant_message_count.items(),
    key=lambda item: item[1],
    reverse=True
)

print("=" * 55)
print("              PARTICIPANT STATISTICS")
print("=" * 55)

# Display message count and percentage contribution
for participant, count in sorted_participants:

    print(
        f"{participant:<15} : "
        f"{count:>4} messages "
        f"({participant_percentage[participant]:.2f}%)"
    )

print()

# Display the most and least active participants
print(f"Most Active Participant  : {most_active} ({highest_messages} messages)")
print(f"Least Active Participant : {least_active} ({lowest_messages} messages)")

print("=" * 55)

              PARTICIPANT STATISTICS
Rahul           :  953 messages (30.03%)
Priya           :  718 messages (22.62%)
Neha            :  635 messages (20.01%)
Aman            :  490 messages (15.44%)
Karan           :  354 messages (11.15%)
Vikas           :   24 messages (0.76%)

Most Active Participant  : Rahul (953 messages)
Least Active Participant : Vikas (24 messages)


## Observation



In [23]:
# ==========================================================
# Observation
# ==========================================================

print(f"{most_active} is the most active participant with {highest_messages} messages.")

print(f"{least_active} is the least active participant with only {lowest_messages} messages.")

print(f"{most_active} contributed {participant_percentage[most_active]:.2f}% of the total user messages.")

Rahul is the most active participant with 953 messages.
Vikas is the least active participant with only 24 messages.
Rahul contributed 30.03% of the total user messages.


# Feature 4 : Activity Analysis

## Objective

The objective of this feature is to analyze the activity of the WhatsApp group by identifying the busiest day of the week and the busiest hour of the day. This helps in understanding the communication pattern of the group members.

In [27]:
# ==========================================================
# Feature 4 : Activity Analysis
# ==========================================================

# Dictionary to store messages sent on each day of the week
day_message_count = {}

# Dictionary to store messages sent during each hour
hour_message_count = {}

# Names of the weekdays
week_days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

# Loop through every parsed message
for message in parsed_messages:

    # Ignore system messages
    if message["sender"] == "System":
        continue

    # Get datetime object
    current_datetime = message["datetime"]

    # Extract weekday name
    day_name = week_days[current_datetime.weekday()]

    # Extract hour
    hour = current_datetime.hour

    # Count messages for each weekday
    if day_name not in day_message_count:
        day_message_count[day_name] = 0

    day_message_count[day_name] += 1

    # Count messages for each hour
    if hour not in hour_message_count:
        hour_message_count[hour] = 0

    hour_message_count[hour] += 1

In [28]:
# ==========================================================
# Find Busiest Day and Hour
# ==========================================================

# Find busiest day
busiest_day = max(day_message_count, key=day_message_count.get)
busiest_day_messages = day_message_count[busiest_day]

# Find busiest hour
busiest_hour = max(hour_message_count, key=hour_message_count.get)
busiest_hour_messages = hour_message_count[busiest_hour]

In [29]:
# ==========================================================
# Display Day-wise Activity
# ==========================================================

print("=" * 55)
print("             DAY-WISE ACTIVITY")
print("=" * 55)

for day in week_days:

    count = day_message_count.get(day, 0)

    print(f"{day:<12} : {count} messages")

print("=" * 55)

             DAY-WISE ACTIVITY
Monday       : 482 messages
Tuesday      : 426 messages
Wednesday    : 483 messages
Thursday     : 475 messages
Friday       : 403 messages
Saturday     : 459 messages
Sunday       : 446 messages


In [30]:
# ==========================================================
# Display Hour-wise Activity
# ==========================================================

print("=" * 55)
print("            HOUR-WISE ACTIVITY")
print("=" * 55)

for hour in range(24):

    count = hour_message_count.get(hour, 0)

    print(f"{hour:02d}:00 - {hour:02d}:59 : {count} messages")

print("=" * 55)

            HOUR-WISE ACTIVITY
00:00 - 00:59 : 57 messages
01:00 - 01:59 : 82 messages
02:00 - 02:59 : 83 messages
03:00 - 03:59 : 77 messages
04:00 - 04:59 : 110 messages
05:00 - 05:59 : 29 messages
06:00 - 06:59 : 33 messages
07:00 - 07:59 : 55 messages
08:00 - 08:59 : 122 messages
09:00 - 09:59 : 151 messages
10:00 - 10:59 : 160 messages
11:00 - 11:59 : 114 messages
12:00 - 12:59 : 193 messages
13:00 - 13:59 : 159 messages
14:00 - 14:59 : 162 messages
15:00 - 15:59 : 131 messages
16:00 - 16:59 : 189 messages
17:00 - 17:59 : 173 messages
18:00 - 18:59 : 248 messages
19:00 - 19:59 : 228 messages
20:00 - 20:59 : 166 messages
21:00 - 21:59 : 177 messages
22:00 - 22:59 : 116 messages
23:00 - 23:59 : 159 messages


In [31]:
# ==========================================================
# Display Summary
# ==========================================================

print("=" * 55)
print("          ACTIVITY SUMMARY")
print("=" * 55)

print(f"Busiest Day  : {busiest_day} ({busiest_day_messages} messages)")
print(f"Busiest Hour : {busiest_hour:02d}:00 - {busiest_hour:02d}:59 ({busiest_hour_messages} messages)")

print("=" * 55)

          ACTIVITY SUMMARY
Busiest Day  : Wednesday (483 messages)
Busiest Hour : 18:00 - 18:59 (248 messages)


## Observation

The activity analysis shows how the messaging activity is distributed across different days of the week and different hours of the day. The busiest day and busiest hour indicate when the group members are most active.

# Feature 5 : NumPy Activity Matrix

## Objective

The objective of this feature is to create a NumPy matrix that stores the messaging activity of every participant for each hour of the day. Each row represents a participant, and each column represents an hour (0–23). This matrix will be used to generate the activity heatmap in the next step.

In [32]:
# ==========================================================
# Feature 5 : NumPy Activity Matrix
# ==========================================================

# Create a sorted list of participant names
participant_names = sorted(participants)

# Create a dictionary to map each participant to a row index
participant_index = {}

for index, name in enumerate(participant_names):
    participant_index[name] = index

# Create a NumPy matrix with rows = participants and columns = 24 hours
activity_matrix = np.zeros((len(participant_names), 24), dtype=int)

# Fill the activity matrix
for message in parsed_messages:

    # Ignore system messages
    if message["sender"] == "System":
        continue

    row = participant_index[message["sender"]]
    column = message["datetime"].hour

    activity_matrix[row][column] += 1

## Matrix Verification

The following code displays the generated activity matrix to verify that the hourly message counts have been stored correctly.

In [33]:
# ==========================================================
# Display Activity Matrix
# ==========================================================

print("Participant Order")

for i, name in enumerate(participant_names):
    print(f"{i} : {name}")

print("\nActivity Matrix\n")

print(activity_matrix)

Participant Order
0 : Aman
1 : Karan
2 : Neha
3 : Priya
4 : Rahul
5 : Vikas

Activity Matrix

[[ 54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7
   16   8  13  11   0  56]
 [  0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27
   25  32  23  14   9   8]
 [  0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47
   62  50  45  27  28  30]
 [  0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40
   38  60  43  32  18   9]
 [  3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49
  105  76  41  92  60  54]
 [  0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3
    2   2   1   1   1   2]]


## Observation

The activity matrix has been successfully created using NumPy. Each row corresponds to one participant, while each column represents one hour of the day. The values stored in the matrix indicate the number of messages sent by each participant during that hour.

# Feature 6 : Text-Based Activity Heatmap

## Objective

The objective of this feature is to visualize the hourly messaging activity of each participant using a text-based heatmap. The heatmap is generated from the NumPy activity matrix without using any external visualization libraries.

In [34]:
# ==========================================================
# Feature 6 : Text-Based Activity Heatmap
# ==========================================================

# This function converts message count into a symbol
def get_heatmap_symbol(message_count):

    if message_count == 0:
        return "."

    elif message_count <= 10:
        return "░"

    elif message_count <= 25:
        return "▒"

    elif message_count <= 50:
        return "▓"

    else:
        return "█"

In [ ]:
# ==========================================================
# Display Heatmap Legend
# ==========================================================

print("=" * 60)
print("HEATMAP LEGEND")
print("=" * 60)

print(".  : No Activity")
print("░  : Low Activity")
print("▒  : Moderate Activity")
print("▓  : High Activity")
print("█  : Very High Activity")

print("=" * 60)

In [36]:
# ==========================================================
# Display Activity Heatmap
# ==========================================================

print("\nTEXT-BASED ACTIVITY HEATMAP\n")

# Print table header
print(f"{'Participant':<12}", end="")

for hour in range(24):
    print(f"{hour:>3}", end="")

print()

print("-" * 85)

# Print one row for each participant
for row in range(len(participant_names)):

    # Print participant name
    print(f"{participant_names[row]:<12}", end="")

    # Print activity symbols
    for column in range(24):

        symbol = get_heatmap_symbol(activity_matrix[row][column])

        print(f"{symbol:>3}", end="")

    print()


TEXT-BASED ACTIVITY HEATMAP

Participant   0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
-------------------------------------------------------------------------------------
Aman          █  █  █  █  █  .  .  .  .  .  .  .  .  .  ▒  ▒  ▒  ░  ▒  ░  ▒  ▒  .  █
Karan         .  .  .  .  .  .  .  ░  ▒  ▒  ▒  ▒  ▓  ▒  ▓  ▓  ▓  ▓  ▒  ▓  ▒  ▒  ░  ░
Neha          .  .  .  .  .  ▒  ░  ▒  ▓  █  █  ▒  ▓  ▓  ▓  ░  ▓  ▓  █  ▓  ▓  ▓  ▓  ▓
Priya         .  .  .  .  .  .  ▒  ▒  ▓  █  █  █  █  ▓  ▓  ▓  ▓  ▓  ▓  █  ▓  ▓  ▒  ░
Rahul         ░  ▒  ▒  ▒  ▒  ░  ▒  ▒  ▒  ▒  ▒  ▒  █  ▓  ▓  █  █  ▓  █  █  ▓  █  █  █
Vikas         .  .  .  .  .  .  .  ░  ░  ░  ░  .  ░  ░  .  ░  ░  ░  ░  ░  ░  ░  ░  ░


## Observation

The heatmap provides a compact representation of participant activity throughout the day. Darker symbols indicate higher message activity, while lighter symbols represent lower activity. This makes it easier to identify communication patterns among different participants.

# Feature 7 : Word Analytics

## Objective

The objective of this feature is to analyze the words used in the WhatsApp group. It calculates the total number of words, identifies the most frequently used words and provides insights into the conversation topics by excluding common stop words.

In [41]:
# ==========================================================
# Feature 7 : Word Analytics
# ==========================================================

# Common words that do not add much meaning
stop_words = {

    # English Articles, Conjunctions and Prepositions
    "the", "a", "an", "and", "or", "but", "if", "so",
    "to", "of", "in", "on", "at", "for", "from", "by",
    "with", "as", "into", "out", "up", "down",

    # Pronouns
    "i", "me", "my", "mine",
    "we", "our", "ours",
    "you", "your", "yours",
    "he", "him", "his",
    "she", "her", "hers",
    "they", "them", "their",
    "it", "its",
    "this", "that", "these", "those",

    # Common Verbs
    "am", "is", "are", "was", "were",
    "be", "been", "being",
    "do", "does", "did",
    "have", "has", "had",
    "will", "would", "can", "could",
    "should", "shall", "may", "might", "must",

    # WhatsApp Words
    "ok", "okay", "haan", "ha",
    "hmm", "hmmm",
    "yes", "no",
    "bro", "bhai", "guys",
    "lol", "lmao",
    "haha", "hehe",
    "hi", "hello", "hey",
    "ya", "yeah", "yo",

    # Hinglish Words
    "hai", "tha", "thi", "the",
    "kya", "nahi", "nahin",
    "kar", "karo", "karna", "karne",
    "kr", "krna",
    "mera", "meri", "mere",
    "tera", "teri", "tere",
    "apna", "apni", "apne",

    # Dataset Specific
    "media", "omitted",
    "message", "messages",
    "deleted"
}

# Dictionary to store frequency of each word
word_frequency = {}

# Variable to count total number of words
total_words = 0

In [42]:
# ==========================================================
# Count Word Frequency
# ==========================================================

for message in parsed_messages:

    # Ignore system messages
    if message["sender"] == "System":
        continue

    # Convert message into lowercase
    text = message["message"].lower()

    # Remove simple punctuation
    punctuation = ".,!?;:()[]{}\"'<>/-"

    for symbol in punctuation:
        text = text.replace(symbol, "")

    # Split message into words
    words = text.split()

    for word in words:

        total_words += 1

        # Ignore stop words
        if word in stop_words:
            continue

        # Ignore media and deleted messages
        if word == "<media":
            continue

        if word == "omitted>":
            continue

        if word == "this":
            continue

        if word == "message":
            continue

        if word == "was":
            continue

        if word == "deleted":
            continue

        # Add new word
        if word not in word_frequency:
            word_frequency[word] = 0

        word_frequency[word] += 1

In [43]:
# ==========================================================
# Find Top 10 Most Frequent Words
# ==========================================================

# Convert dictionary into list
word_list = list(word_frequency.items())

# Sort according to frequency
word_list.sort(key=lambda item: item[1], reverse=True)

top_words = word_list[:10]

In [44]:
# ==========================================================
# Display Word Analytics
# ==========================================================

print("=" * 55)
print("              WORD ANALYTICS")
print("=" * 55)

print(f"Total Words : {total_words}")

print("\nTop 10 Most Frequent Words\n")

for word, frequency in top_words:

    print(f"{word:<15} : {frequency}")

print("=" * 55)

              WORD ANALYTICS
Total Words : 31551

Top 10 Most Frequent Words

how             : 321
about           : 274
today           : 257
just            : 208
which           : 202
everyone        : 187
telling         : 179
one             : 157
started         : 150
scene           : 145


## Observation

This feature analyzes the vocabulary used in the WhatsApp group. Common filler words are ignored so that the analysis highlights meaningful words that appear frequently in the conversation.

# Feature 8 : Response Speed Analysis

## Objective

The objective of this feature is to calculate the average response time for every participant. A response is counted only when a participant replies after another participant. This helps identify the fastest and slowest repliers in the group.

In [51]:
# ==========================================================
# Feature 8 : Response Speed Analysis
# ==========================================================

# Dictionary to store total response time (minutes)
response_time = {}

# Dictionary to store number of valid replies
response_count = {}

# Initialize dictionaries
for person in participants:

    response_time[person] = 0
    response_count[person] = 0


# Compare consecutive messages
for i in range(1, len(parsed_messages)):

    previous = parsed_messages[i - 1]
    current = parsed_messages[i]

    # Ignore system messages
    if previous["sender"] == "System":
        continue

    if current["sender"] == "System":
        continue

    # Ignore consecutive messages from same sender
    if previous["sender"] == current["sender"]:
        continue

    # Calculate response gap
    gap = current["datetime"] - previous["datetime"]

    minutes = gap.total_seconds() / 60

    if minutes < 0:
        continue

    responder = current["sender"]

    response_time[responder] += minutes
    response_count[responder] += 1

In [52]:
# ==========================================================
# Calculate Average Response Time
# ==========================================================

average_response = {}

for person in participants:

    if response_count[person] > 0:

        average_response[person] = (
            response_time[person] /
            response_count[person]
        )

    else:

        average_response[person] = 0

In [55]:
# ==========================================================
# Display Response Speed
# ==========================================================

print("=" * 65)
print("                 RESPONSE SPEED ANALYSIS")
print("=" * 65)

fastest_person = None
slowest_person = None

fastest_time = float("inf")
slowest_time = -1

for person in sorted(participants):

    avg = average_response[person]

    hours = int(avg // 60)
    minutes = avg % 60

    if hours > 0:
        display_time = f"{hours} hr {minutes:.2f} min"
    else:
        display_time = f"{minutes:.2f} min"

    print(f"{person:<10} : {display_time}")

    if avg > 0 and avg < fastest_time:
        fastest_time = avg
        fastest_person = person

    if avg > slowest_time:
        slowest_time = avg
        slowest_person = person

print("\n" + "=" * 65)

fast_h = int(fastest_time // 60)
fast_m = fastest_time % 60

if fast_h > 0:
    fast_display = f"{fast_h} hr {fast_m:.2f} min"
else:
    fast_display = f"{fast_m:.2f} min"

slow_h = int(slowest_time // 60)
slow_m = slowest_time % 60

if slow_h > 0:
    slow_display = f"{slow_h} hr {slow_m:.2f} min"
else:
    slow_display = f"{slow_m:.2f} min"

print(f"Fastest Replier : {fastest_person} ({fast_display})")
print(f"Slowest Replier : {slowest_person} ({slow_display})")

print("=" * 65)

                 RESPONSE SPEED ANALYSIS
Aman       : 55.36 min
Karan      : 36.23 min
Neha       : 39.45 min
Priya      : 41.99 min
Rahul      : 34.95 min
Vikas      : 46.30 min

Fastest Replier : Rahul (34.95 min)
Slowest Replier : Aman (55.36 min)


## Observation

The response speed analysis calculates the average time taken by each participant to reply after another participant's message. The participant with the smallest average response time is considered the fastest replier, while the participant with the largest average response time is considered the slowest replier.

# Feature 9 : Participant Silent Streak Analysis

## Objective

The objective of this feature is to calculate the longest silent period for each participant. A silent streak represents the maximum time gap between two consecutive messages sent by the same participant.

In [56]:
# ==========================================================
# Feature 9 : Participant Silent Streak Analysis
# ==========================================================

# Dictionary to store longest silent streak (in minutes)
silent_streak = {}

# Dictionary to store previous message time
last_message_time = {}

# Initialize dictionaries
for person in participants:

    silent_streak[person] = 0
    last_message_time[person] = None


# Traverse all messages
for message in parsed_messages:

    sender = message["sender"]

    # Ignore system messages
    if sender == "System":
        continue

    current_time = message["datetime"]

    # First message of participant
    if last_message_time[sender] is None:

        last_message_time[sender] = current_time
        continue

    # Calculate time gap
    gap = current_time - last_message_time[sender]

    gap_minutes = gap.total_seconds() / 60

    # Update longest gap
    if gap_minutes > silent_streak[sender]:

        silent_streak[sender] = gap_minutes

    # Update latest message time
    last_message_time[sender] = current_time

In [57]:
# ==========================================================
# Display Silent Streak Analysis
# ==========================================================

print("=" * 65)
print("             PARTICIPANT SILENT STREAKS")
print("=" * 65)

longest_person = None
largest_gap = -1

for person in sorted(participants):

    total_minutes = int(silent_streak[person])

    days = total_minutes // (24 * 60)

    remaining = total_minutes % (24 * 60)

    hours = remaining // 60

    minutes = remaining % 60

    print(
        f"{person:<10} : "
        f"{days} day(s), "
        f"{hours} hr(s), "
        f"{minutes} min"
    )

    if silent_streak[person] > largest_gap:

        largest_gap = silent_streak[person]
        longest_person = person


print("\n" + "=" * 65)

gap_minutes = int(largest_gap)

days = gap_minutes // (24 * 60)

remaining = gap_minutes % (24 * 60)

hours = remaining // 60

minutes = remaining % 60

print(
    f"Longest Silent Participant : "
    f"{longest_person} "
    f"({days} day(s), {hours} hr(s), {minutes} min)"
)

print("=" * 65)

             PARTICIPANT SILENT STREAKS
Aman       : 0 day(s), 21 hr(s), 51 min
Karan      : 0 day(s), 20 hr(s), 42 min
Neha       : 1 day(s), 4 hr(s), 22 min
Priya      : 0 day(s), 15 hr(s), 47 min
Rahul      : 1 day(s), 0 hr(s), 40 min
Vikas      : 12 day(s), 4 hr(s), 38 min

Longest Silent Participant : Vikas (12 day(s), 4 hr(s), 38 min)


## Observation

This analysis calculates the maximum inactivity period for every participant by measuring the time gap between two consecutive messages sent by the same participant. A larger silent streak indicates that the participant remained inactive for a longer duration before sending another message.

# Feature 10 : Personality Archetypes

## Objective

The objective of this feature is to identify the personality archetype of each participant based on their messaging behaviour. The classification is performed using the statistics calculated in the previous features such as message count, activity pattern, response speed and silent streak.

In [58]:
# ==========================================================
# Feature 10 : Personality Archetypes
# ==========================================================

# Dictionary to store total questions asked
question_count = {}

# Dictionary to store total words typed
word_count = {}

# Dictionary to store total messages
message_count = {}

# Dictionary to store late night messages
night_messages = {}

# Dictionary to store laughter words
laughter_count = {}

# Initialize dictionaries
for person in participants:

    question_count[person] = 0
    word_count[person] = 0
    message_count[person] = 0
    night_messages[person] = 0
    laughter_count[person] = 0

In [59]:
# ==========================================================
# Collect Personality Statistics
# ==========================================================

for message in parsed_messages:

    sender = message["sender"]

    if sender == "System":
        continue

    text = message["message"].lower()

    message_count[sender] += 1

    # Count words
    words = text.split()

    word_count[sender] += len(words)

    # Count questions
    question_count[sender] += text.count("?")

    # Count late night messages
    hour = message["datetime"].hour

    if hour >= 22 or hour <= 5:
        night_messages[sender] += 1

    # Count laughter words
    laughter_words = [
        "haha",
        "hehe",
        "lol",
        "lmao",
        "😂",
        "🤣"
    ]

    for laugh in laughter_words:

        if laugh in text:
            laughter_count[sender] += 1

In [60]:
# ==========================================================
# Calculate Average Message Length
# ==========================================================

average_words = {}

for person in participants:

    if message_count[person] > 0:

        average_words[person] = (
            word_count[person] /
            message_count[person]
        )

    else:

        average_words[person] = 0

In [63]:
# ==========================================================
# Assign Personality Archetypes
# ==========================================================

# Dictionary to store caring keyword score
caring_score = {}

# Dictionary to store drama score
drama_score = {}

# Initialize
for person in participants:

    caring_score[person] = 0
    drama_score[person] = 0


# Caring keywords for Group Mom
caring_words = [
    "okay",
    "ok",
    "safe",
    "eat",
    "sleep",
    "take care",
    "please",
    "drink water",
    "reminder",
    "don't forget",
    "dont forget",
    "are you"
]


# Calculate scores
for message in parsed_messages:

    sender = message["sender"]

    if sender == "System":
        continue

    text = message["message"]

    lower_text = text.lower()

    # Group Mom Score
    for word in caring_words:

        if word in lower_text:
            caring_score[sender] += 1

    # Drama Queen Score

    words = text.split()

    upper_words = 0

    for w in words:

        if len(w) > 2 and w.isupper():

            upper_words += 1

    if upper_words >= 2:

        drama_score[sender] += 1

    if text.count("!") >= 2:

        drama_score[sender] += 1


personality = {}

# Rahul -> Spammer
spammer = max(message_count, key=message_count.get)
personality[spammer] = "THE SPAMMER"

# Priya -> Group Mom
group_mom = max(caring_score, key=caring_score.get)
personality[group_mom] = "THE GROUP MOM"

# Aman -> Night Owl
night_owl = max(night_messages, key=night_messages.get)
personality[night_owl] = "THE NIGHT OWL"

# Karan -> Storyteller
storyteller = max(average_words, key=average_words.get)
personality[storyteller] = "THE STORYTELLER"

# Neha -> Drama Queen
drama = max(drama_score, key=drama_score.get)
personality[drama] = "THE DRAMA QUEEN"

# Vikas -> Ghost
ghost = max(silent_streak, key=silent_streak.get)
personality[ghost] = "THE GHOST"

In [64]:
# ==========================================================
# Display Personality Archetypes
# ==========================================================

print("=" * 65)
print("                PERSONALITY ARCHETYPES")
print("=" * 65)

for person in sorted(participants):

    print(f"{person:<10} : {personality[person]}")

print("=" * 65)

                PERSONALITY ARCHETYPES
Aman       : THE NIGHT OWL
Karan      : THE STORYTELLER
Neha       : THE DRAMA QUEEN
Priya      : THE GROUP MOM
Rahul      : THE SPAMMER
Vikas      : THE GHOST


## Observation

The personality archetypes are assigned using measurable statistics extracted from the WhatsApp chat. The classification is based on messaging frequency, response behaviour, question asking, average message length, late-night activity and conversational style.

# Final Summary Report

## Objective

The objective of this section is to summarize all the important insights obtained from the WhatsApp group chat analysis. This report highlights participant activity, communication patterns, response behaviour and personality archetypes in a concise and readable format.

In [66]:
# ==========================================================
# Final Summary Report
# ==========================================================

print("=" * 70)
print("                 WHATSAPP GROUP CHAT ANALYSIS REPORT")
print("=" * 70)

print("\nGENERAL INFORMATION")
print("-" * 70)
print(f"Total Messages           : {user_messages}")
print(f"Total Participants       : {len(participants)}")
print(f"Chat Duration : {chat_duration.days} Days")
print(f"Media Messages           : {media_messages}")
print(f"Deleted Messages         : {deleted_messages}")

print("\nPARTICIPANT INSIGHTS")
print("-" * 70)
print(f"Most Active Participant  : {most_active}")
print(f"Least Active Participant : {least_active}")

print("\nACTIVITY ANALYSIS")
print("-" * 70)
print(f"Busiest Day              : {busiest_day}")
print(f"Busiest Hour             : {busiest_hour}:00 - {busiest_hour}:59")

print("\nCOMMUNICATION ANALYSIS")
print("-" * 70)
print(f"Fastest Replier          : {fastest_person}")
print(f"Slowest Replier          : {slowest_person}")
print(f"Longest Silent Member    : {longest_person}")

print("\nPERSONALITY ARCHETYPES")
print("-" * 70)

for person in sorted(participants):
    print(f"{person:<10} : {personality[person]}")

print("\nCONCLUSION")
print("-" * 70)
print("The WhatsApp group analysis successfully identified")
print("participant activity, communication behaviour,")
print("response patterns, active hours and personality")
print("archetypes using Python fundamentals and NumPy.")
print("The project demonstrates practical data analysis")
print("without using high-level data analysis libraries.")

print("\n" + "=" * 70)
print("                 END OF REPORT")
print("=" * 70)

                 WHATSAPP GROUP CHAT ANALYSIS REPORT

GENERAL INFORMATION
----------------------------------------------------------------------
Total Messages           : 3174
Total Participants       : 6
Chat Duration : 59 Days
Media Messages           : 32
Deleted Messages         : 15

PARTICIPANT INSIGHTS
----------------------------------------------------------------------
Most Active Participant  : Rahul
Least Active Participant : Vikas

ACTIVITY ANALYSIS
----------------------------------------------------------------------
Busiest Day              : Wednesday
Busiest Hour             : 18:00 - 18:59

COMMUNICATION ANALYSIS
----------------------------------------------------------------------
Fastest Replier          : Rahul
Slowest Replier          : Aman
Longest Silent Member    : Vikas

PERSONALITY ARCHETYPES
----------------------------------------------------------------------
Aman       : THE NIGHT OWL
Karan      : THE STORYTELLER
Neha       : THE DRAMA QUEEN
Priya     

## Observation

The final summary combines all the results obtained from previous features into a single report. It provides a quick overview of group behaviour, participant activity, communication trends and personality archetypes, making the overall analysis easy to understand.